# 05 · Compare Trilingua against other small open models

Bits-per-byte on identical held-out EN/ES/FR text (tokenizer-independent), 1024-token window for every model. **Inputs:** outputs of 01 and 02. GPU + Internet on.

In [ ]:
import os, sys, glob, subprocess, json
CODE = "/tmp/trilingua"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/aladewealth/Trilingual-.git", CODE], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
metas = glob.glob("/kaggle/input/**/meta.json", recursive=True)
DATA = os.path.dirname(metas[0])
from trilingua.utils import find_latest_checkpoint
BASE = find_latest_checkpoint(['/kaggle/input'])
print("DATA", DATA, "\nBASE", BASE)

In [ ]:
MODELS = ["gpt2", "EleutherAI/pythia-70m", "EleutherAI/pythia-160m", "HuggingFaceTB/SmolLM2-135M",
          "bigscience/bloom-560m", "Qwen/Qwen2.5-0.5B"]
!python -m trilingua.evaluate --model {BASE} --data_dir {DATA} --out_dir /kaggle/working/compare --tag trilingua \
    --compare {" ".join(MODELS)} --n_docs 200 --langid_n 10 --device cuda

In [ ]:
r = json.load(open("/kaggle/working/compare/eval_trilingua.json"))
bpb, npar = r["doc_bpb"], r.get("compare_params", {})
npar["trilingua"] = r["params"]
print(f"{'model':32s} {'params':>8s} {'en':>6s} {'es':>6s} {'fr':>6s} {'avg':>6s}")
rows = []
for name, v in bpb.items():
    avg = sum(v.values()) / len(v)
    rows.append((avg, name, v))
for avg, name, v in sorted(rows):
    print(f"{name:32s} {npar.get(name, 0)/1e6:7.0f}M {v['en']:6.3f} {v['es']:6.3f} {v['fr']:6.3f} {avg:6.3f}")
json.dump({"bpb": bpb, "params": npar}, open("/kaggle/working/compare/compare_table.json", "w"), indent=2)

In [ ]:
from IPython.display import Image
Image("/kaggle/working/compare/bpb_trilingua.png")